In [1]:
import os
import shutil
from dotenv import load_dotenv
import gradio as gr

In [2]:
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [3]:
MODEL = "gpt-4o-mini"
DB_DIR = "vector_db"

In [4]:
load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your-api-key")

In [7]:
folders = ["knowledge-base"]
documents = []


for folder in folders:
    loader = DirectoryLoader(
        folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    documents.extend(loader.load())


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"✅ Loaded {len(chunks)} chunks from knowledge base.")

✅ Loaded 10 chunks from knowledge base.


In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# Delete if already exists

# Reset vector db
if os.path.exists(DB_DIR):
   shutil.rmtree(DB_DIR)


vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_DIR)
print(f"✅ Vectorstore created with {vectorstore._collection.count()} documents")

✅ Vectorstore created with 10 documents


In [7]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 4 vectors with 1,536 dimensions in the vector store


In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
qa_chain = ConversationalRetrievalChain.from_llm(
ChatOpenAI(model=MODEL, temperature=0),
retriever=retriever,
memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True),
)

C:\Users\Hi-Tech\AppData\Local\Temp\ipykernel_5584\1347783759.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True),


In [12]:
def respond(user_message, history):
    result = qa_chain.invoke({"question": user_message})
    history = history + [(user_message, result["answer"])]
    return history, ""

In [15]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🤖 ACM NUCES Knowledge Assistant")
    with gr.Row():
      with gr.Column(scale=3):
          chatbot = gr.Chatbot(height=500, label="Chat with ACM Knowledge Base")
          msg = gr.Textbox(placeholder="Ask about Coders Cup, Developers Day, SkillPrep, or Teams...", label="Your Question")
          submit = gr.Button("Ask", variant="primary")
      with gr.Column(scale=1):
           gr.Markdown("### ℹ️ Info\nAsk about:\n- Events (Coders Cup, Developers Day)\n- Programs (SkillPrep)\n- Teams & Organization")


    submit.click(respond, [msg, chatbot], [chatbot, msg])
    msg.submit(respond, [msg, chatbot], [chatbot, msg])


demo.launch()

C:\Users\Hi-Tech\AppData\Local\Temp\ipykernel_5584\1042994136.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500, label="Chat with ACM Knowledge Base")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
